In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


In [2]:
import cv2 as cv
import numpy as np
from matplotlib.widgets import Slider, Button
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('TkAgg')


In [3]:
# Loading the image in grayscale
img = cv.imread("bone-scan.tif", cv.IMREAD_GRAYSCALE)

# Defining the kernel size for median blur
kernel_size = 3
# Applying median blur to the image
median_blurred = cv.medianBlur(img, kernel_size)


In [4]:
# Defining a function for image sharpening using Laplacian
def sharpening_func(image, c):
    # Calculating the Laplacian
    laplacian = cv.Laplacian(image, cv.CV_64F)

    # Converting the result to an 8-bit image
    laplacian = cv.convertScaleAbs(laplacian)

    # Adding Laplacian to the original image with a scaling factor c
    img = image.copy() + c * laplacian

    return img


In [5]:
# Initializing the sharpening parameter c
c0 = 0
# Defining the step size for parameter variation
delta_c = 0.5
# Sharpening the image using the sharpening function
sharpened_img = sharpening_func(median_blurred, c0)

# Calculating the histogram of the sharpened image
histogram, _ = np.histogram(sharpened_img, bins=256, range=(0, 255))

# Creating a Matplotlib figure with two subplots
fig, ax = plt.subplots(1, 2)
plt.suptitle("Adding Laplacian to Original Image via c Parameter Variation")
plt.subplots_adjust(bottom=0.25, top=0.9)

# Displaying the sharpened image in the first subplot
l_img = ax[0].imshow(sharpened_img, cmap="gray", vmin=0, vmax=255)
ax[0].set_axis_off()
ax[0].set_title("Sharpened Image")

# Displaying the histogram in the second subplot
l_hist, = ax[1].plot(histogram)
plt.yscale("log")
ax[1].set_title("Histogram")
ax[1].margins(x=0)

# Defining the color for the slider
axcolor = 'lightgoldenrodyellow'
# Creating a slider for adjusting the parameter c
ax_c = plt.axes([0.25, 0.1, 0.65, 0.03], facecolor=axcolor)
s_c = Slider(ax_c, 'c', -20.0, 20.0, valinit=c0, valstep=delta_c)

# Defining a function to update the image and histogram based on the slider value
def update(val):
    c = s_c.val
    ax[0].imshow(sharpening_func(median_blurred, c), cmap="gray", vmin=0, vmax=255)
    histogram, _ = np.histogram(sharpening_func(median_blurred, c), bins=256, range=(0, 255))
    ax[1].lines.pop(-1)
    l_hist, = ax[1].plot(histogram, "C0")
    plt.yscale("log")
    fig.canvas.draw_idle()

# Attaching the update function to the slider
s_c.on_changed(update)

# Creating a reset button
resetax = plt.axes([0.8, 0.025, 0.1, 0.04])
button = Button(resetax, 'Reset', color=axcolor, hovercolor='0.975')

# Defining a function to reset the slider value
def reset(event):
    s_c.reset()

# Attaching the reset function to the button
button.on_clicked(reset)

# Displaying the Matplotlib plot
plt.show()


Text(0.5, 0.98, 'Adding Laplacian to Original Image via c Parameter Variation')

Text(0.5, 1.0, 'Sharpened Image')

Text(0.5, 1.0, 'Histogram')

0

0